# Compare DP and Q-Learning

Notebook version of `compare_dp_q_learning.py`.

## Imports

This notebook pulls in the dynamic-programming solver, the Q-learning solver, and the shared gridworld formatter. Keeping the imports local makes the comparison notebook small while still reusing the validated implementations.


In [ ]:
from __future__ import annotations

import numpy as np

from dynamic_programming_case_study import policy_iteration_with_history
from gridworld_case_study import GridWorldCaseStudyEnv, format_policy
from q_learning_case_study import q_learning_with_history


## Snapshot Printers

The helper functions below format intermediate outputs from both algorithms into comparable 3x3 tables. That makes it easy to inspect how dynamic programming converges through policy iteration while Q-learning improves through episodes of sampled experience.


In [ ]:
def print_dp_snapshots(dp_history):
    print("Dynamic Programming Snapshots")
    for snapshot in dp_history:
        print(
            f"\nIteration {snapshot['iteration']} "
            f"(stable={snapshot['stable']})"
        )
        print("V")
        print(snapshot["V"].reshape(3, 3))
        print("Greedy policy")
        print(format_policy(snapshot["policy"]))


def print_q_learning_snapshots(ql_snapshots):
    print("\nQ-Learning Snapshots")
    for snapshot in ql_snapshots:
        q_values = snapshot["Q"]
        print(
            f"\nEpisode {snapshot['episode']} "
            f"(epsilon={snapshot['epsilon']:.3f})"
        )
        print("max_a Q(s, a)")
        print(q_values.max(axis=1).reshape(3, 3))
        print("Greedy policy")
        print(format_policy(snapshot["policy"]))


## Run The Comparison

This final section executes both methods on the same environment and prints their final value estimates, policies, and absolute Q-table differences. It is a direct side-by-side view of model-based planning versus model-free learning on the identical MDP.


In [ ]:
env = GridWorldCaseStudyEnv()

dp_policy, dp_v, dp_q, dp_history = policy_iteration_with_history(env)
ql_q, ql_snapshots = q_learning_with_history(env)
ql_policy = ql_snapshots[-1]["policy"]
ql_v = ql_q.max(axis=1)

np.set_printoptions(precision=3, suppress=True)

print_dp_snapshots(dp_history)
print_q_learning_snapshots(ql_snapshots)

print("\nFinal Comparison")
print("\nDynamic Programming: V*")
print(dp_v.reshape(3, 3))
print("\nQ-Learning: max_a Q(s, a)")
print(ql_v.reshape(3, 3))

print("\nDynamic Programming Policy")
print(format_policy(dp_policy))
print("\nQ-Learning Policy")
print(format_policy(ql_policy))

print("\nPolicy match by state")
print((dp_policy.argmax(axis=1) == ql_policy.argmax(axis=1)).reshape(3, 3))

diff = np.abs(dp_q - ql_q)
print("\nAbsolute Q-table difference |Q*_DP - Q_QL|")
print(diff)
print(f"\nMean absolute difference: {diff.mean():.4f}")
print(f"Max absolute difference: {diff.max():.4f}")
